In [ ]:
model_desc = """
    # Perspectiva de Processos ← Aprendizado
    P1 ~ A1 + A2 + A4
    P2 ~ A1 + A3 + P1
    P3 ~ P1
    P4 ~ A2

    # Perspectiva de Clientes ← Processos
    C1 ~ P1 + P2 + P3 + P4
    C2 ~ C1
    C3 ~ C1
    C4 ~ P1 + P2

    # Perspectiva Financeira ← Clientes + Processos
    F1 ~ C2 + C3
    F2 ~ F1 + P1
    F3 ~ F2
    F4 ~ C4
"""

In [ ]:
# Covariâncias dentro da mesma perspectiva (opcional — melhora o ajuste)
    A1 ~~ A2    # treinamento e invest. tecnologia podem ser correlacionados
    A1 ~~ A3    # empresas que treinam mais tendem a ter mais engajamento
    P1 ~~ P3    # eficiência e pontualidade co-variam
    C2 ~~ C3    # retenção e wallet share são faces da mesma fidelidade

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import semopy
import matplotlib.pyplot as plt
import networkx as nx
import matplotlib.patches as mpatches

# ════════════════════════════════════════════════════════════════
# BLOCO 1 — Dataset (simulado com os 16 KPIs do BSC real)
# ════════════════════════════════════════════════════════════════
np.random.seed(42)
N = 200

def zscale(x): return (x - x.mean()) / x.std()

# Aprendizado & Crescimento (causas raiz)
A1 = np.random.normal(40,   8,   N)   # horas treinamento/colaborador
A2 = np.random.normal(8.5,  1.5, N)   # investimento TI (% receita)
A3 = np.random.normal(28,   12,  N)   # eNPS colaboradores
A4 = np.random.normal(16,   4,   N)   # turnover voluntário (%)

# Processos (dependem de Aprendizado)
P1 = np.clip(0.75 + 0.35*zscale(A1) + 0.28*zscale(A2)
             - 0.20*zscale(A4) + np.random.normal(0, 0.05, N), 0.40, 1.00)

P2 = np.clip(0.15 - 0.45*zscale(P1) - 0.22*zscale(A1)
             - 0.18*zscale(A3) + np.random.normal(0, 0.02, N), 0.01, 0.50)

P3 = np.clip(0.70 + 0.55*zscale(P1)
             + np.random.normal(0, 0.04, N), 0.40, 1.00)

P4 = np.clip(0.50 + 0.40*zscale(A2)
             + np.random.normal(0, 0.05, N), 0.20, 1.00)

# Clientes (dependem de Processos)
C1 = np.clip(45 + 18*zscale(P1) - 12*zscale(P2)
             + 10*zscale(P3) + 8*zscale(P4)
             + np.random.normal(0, 4, N), 0, 100)    # NPS

C2 = np.clip(0.88 + 0.35*zscale(C1)
             + np.random.normal(0, 0.02, N), 0.50, 1.00)  # retenção

C3 = np.clip(400 + 120*zscale(C1)
             + np.random.normal(0, 30, N), 100, 900)       # wallet share R$k

C4 = np.clip(12 - 4*zscale(P1) - 3*zscale(P2)
             + np.random.normal(0, 1.5, N), 1, 48)         # MTTR horas

# Financeiro (dependem de Clientes + Processos)
F1 = np.clip(48 + 15*zscale(C2) + 10*zscale(C3)
             + np.random.normal(0, 4, N), 20, 100)          # receita R$MM

F2 = np.clip(0.22 + 0.08*zscale(F1) + 0.06*zscale(P1)
             + np.random.normal(0, 0.02, N), 0.05, 0.45)   # margem EBITDA

F3 = np.clip(0.45 + 0.30*zscale(F2)
             + np.random.normal(0, 0.04, N), 0.10, 0.90)   # MRR (%)

F4 = np.clip(0.035 - 0.015*zscale(C4)
             + np.random.normal(0, 0.005, N), 0.005, 0.10) # inadimplência

df = pd.DataFrame({
    'A1':A1, 'A2':A2, 'A3':A3, 'A4':A4,
    'P1':P1, 'P2':P2, 'P3':P3, 'P4':P4,
    'C1':C1, 'C2':C2, 'C3':C3, 'C4':C4,
    'F1':F1, 'F2':F2, 'F3':F3, 'F4':F4,
})

# Padronizar
scaler = StandardScaler()
df_s = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)

print(f"Dataset: {df.shape[0]} obs × {df.shape[1]} variáveis")

# ════════════════════════════════════════════════════════════════
# BLOCO 2 — Especificação do modelo SEM
# ════════════════════════════════════════════════════════════════
# Cada linha = uma equação causal do BSC
# O semopy vai estimar um β para cada preditor

model_desc = """
    P1 ~ A1 + A2 + A4
    P2 ~ A1 + A3 + P1
    P3 ~ P1
    P4 ~ A2
    C1 ~ P1 + P2 + P3 + P4
    C2 ~ C1
    C3 ~ C1
    C4 ~ P1 + P2
    F1 ~ C2 + C3
    F2 ~ F1 + P1
    F3 ~ F2
    F4 ~ C4
"""

# ════════════════════════════════════════════════════════════════
# BLOCO 3 — Estimação
# ════════════════════════════════════════════════════════════════
model = semopy.Model(model_desc)
result = model.fit(df_s)           # MLE por default

print("\nResultado da otimização:")
print(result)

# ════════════════════════════════════════════════════════════════
# BLOCO 4 — Parâmetros estimados
# ════════════════════════════════════════════════════════════════
params = model.inspect(mode='list')
print("\nCoeficientes estimados (relações causais):")
print(params[params['op'] == '~'].to_string(index=False))

# ════════════════════════════════════════════════════════════════
# BLOCO 5 — Índices de ajuste do modelo
# ════════════════════════════════════════════════════════════════
stats = semopy.calc_stats(model)
print("\n--- Índices de Ajuste ---")
print(stats.T)

# Interpretação automática
cfi   = float(stats.loc['CFI'].values[0])
rmsea = float(stats.loc['RMSEA'].values[0])
srmr  = float(stats.loc['SRMR'].values[0])

print("\n--- Diagnóstico ---")
print(f"CFI   = {cfi:.3f}   {'✓ BOM (≥ 0.90)'   if cfi >= 0.90   else '✗ RUIM (< 0.90)'}")
print(f"RMSEA = {rmsea:.3f}   {'✓ BOM (≤ 0.08)'   if rmsea <= 0.08 else '✗ RUIM (> 0.08)'}")
print(f"SRMR  = {srmr:.3f}   {'✓ BOM (≤ 0.08)'   if srmr <= 0.08  else '✗ RUIM (> 0.08)'}")

# ════════════════════════════════════════════════════════════════
# BLOCO 6 — Montar o dict do grafo com pesos SEM
# ════════════════════════════════════════════════════════════════
path_df = params[params['op'] == '~'][['rval', 'lval', 'Estimate', 'p-value']]

NODES = {
    'A1': {'perspectiva': 'Aprendizado', 'variavel': 'Horas de Treinamento'},
    'A2': {'perspectiva': 'Aprendizado', 'variavel': 'Investimento em TI (%)'},
    'A3': {'perspectiva': 'Aprendizado', 'variavel': 'eNPS Colaboradores'},
    'A4': {'perspectiva': 'Aprendizado', 'variavel': 'Turnover Voluntário'},
    'P1': {'perspectiva': 'Processos',   'variavel': 'Eficiência de Entrega'},
    'P2': {'perspectiva': 'Processos',   'variavel': 'Taxa de Retrabalho'},
    'P3': {'perspectiva': 'Processos',   'variavel': 'Pontualidade de Entrega'},
    'P4': {'perspectiva': 'Processos',   'variavel': 'Cobertura de Testes'},
    'C1': {'perspectiva': 'Clientes',    'variavel': 'NPS'},
    'C2': {'perspectiva': 'Clientes',    'variavel': 'Taxa de Retenção'},
    'C3': {'perspectiva': 'Clientes',    'variavel': 'Wallet Share'},
    'C4': {'perspectiva': 'Clientes',    'variavel': 'MTTR Suporte'},
    'F1': {'perspectiva': 'Financeiro',  'variavel': 'Receita Total'},
    'F2': {'perspectiva': 'Financeiro',  'variavel': 'Margem EBITDA'},
    'F3': {'perspectiva': 'Financeiro',  'variavel': 'MRR (%)'},
    'F4': {'perspectiva': 'Financeiro',  'variavel': 'Inadimplência'},
}

bsc_graph = {
    'nodes': NODES,
    'edges': [
        {
            'from':    row['rval'],
            'to':      row['lval'],
            'weight':  round(row['Estimate'], 3),
            'p_value': round(row['p-value'], 4),
            'sig':     '***' if row['p-value'] < 0.001
                       else '**'  if row['p-value'] < 0.01
                       else '*'   if row['p-value'] < 0.05
                       else 'n.s.'
        }
        for _, row in path_df.iterrows()
    ]
}

import pprint
pprint.pprint(bsc_graph, sort_dicts=False)

# ════════════════════════════════════════════════════════════════
# BLOCO 7 — Visualização do grafo com pesos SEM
# ════════════════════════════════════════════════════════════════
DARK   = '#070910'
DARK2  = '#0d1017'
DARK3  = '#111620'
BORDER = '#1e2535'
TEXT   = '#e2e8f0'
MUTED  = '#4a5568'

COLOR_MAP = {
    'Aprendizado': '#3b82f6',
    'Processos':   '#8b5cf6',
    'Clientes':    '#10b981',
    'Financeiro':  '#f59e0b',
}

pos = {
    'A1': (0, 5), 'A2': (0, 3.3), 'A3': (0, 1.7), 'A4': (0, 0),
    'P1': (2.5, 4.5), 'P2': (2.5, 3), 'P3': (2.5, 1.5), 'P4': (2.5, 0),
    'C1': (5, 4.5), 'C2': (5, 3), 'C3': (5, 1.5), 'C4': (5, 0),
    'F1': (7.5, 4.5), 'F2': (7.5, 3), 'F3': (7.5, 1.5), 'F4': (7.5, 0),
}

G = nx.DiGraph()
for node, attr in NODES.items():
    G.add_node(node, **attr)
for e in bsc_graph['edges']:
    G.add_edge(e['from'], e['to'], weight=e['weight'],
               sig=e['sig'], p=e['p_value'])

fig, ax = plt.subplots(figsize=(18, 9))
fig.patch.set_facecolor(DARK)
ax.set_facecolor(DARK2)

# Zonas
zones = [
    ('Aprendizado\n& Crescimento', (-0.9,-0.8), 1.8, 6.6, '#3b82f60d'),
    ('Processos\nInternos',        ( 1.7,-0.8), 1.8, 6.6, '#8b5cf60d'),
    ('Clientes',                   ( 4.2,-0.8), 1.8, 6.6, '#10b9810d'),
    ('Financeiro',                 ( 6.7,-0.8), 1.8, 6.6, '#f59e0b0d'),
]
for label, (x,y), w, h, color in zones:
    ax.add_patch(plt.Rectangle((x,y), w, h, linewidth=0.5,
                               edgecolor=BORDER, facecolor=color, zorder=0))
    ax.text(x+w/2, y+h+0.1, label, ha='center',
            color=MUTED, fontsize=8, style='italic', linespacing=1.4)

node_colors = [COLOR_MAP[NODES[n]['perspectiva']] for n in G.nodes()]
nx.draw_networkx_nodes(G, pos, node_color=node_colors,
                       node_size=2000, alpha=0.95, ax=ax,
                       linewidths=1.5, edgecolors='#ffffff22')
nx.draw_networkx_labels(G, pos, font_color='white',
                        font_size=11, font_weight='bold', ax=ax)

edge_list   = list(G.edges())
edge_betas  = [G[u][v]['weight'] for u,v in edge_list]
edge_widths = [abs(b)*6 + 0.5 for b in edge_betas]
edge_colors = ['#ef4444' if b < 0 else '#4a9eff' for b in edge_betas]

for (u,v), width, color in zip(edge_list, edge_widths, edge_colors):
    nx.draw_networkx_edges(G, pos, edgelist=[(u,v)],
                           edge_color=color, width=width, alpha=0.70,
                           arrows=True, arrowsize=15, ax=ax,
                           connectionstyle='arc3,rad=0.08',
                           min_source_margin=26, min_target_margin=26)

edge_labels = {
    (u,v): f"{G[u][v]['weight']:+.2f}{G[u][v]['sig']}"
    for u,v in G.edges()
}
nx.draw_networkx_edge_labels(
    G, pos, edge_labels=edge_labels,
    font_size=7, font_color='#cbd5e1',
    bbox=dict(boxstyle='round,pad=0.2', facecolor=DARK3,
              edgecolor=BORDER, alpha=0.85), ax=ax
)

legend_handles = [mpatches.Patch(color=c, label=p)
                  for p, c in COLOR_MAP.items()]
legend_handles += [
    mpatches.Patch(color='#4a9eff', label='β positivo'),
    mpatches.Patch(color='#ef4444', label='β negativo'),
]
ax.legend(handles=legend_handles, loc='lower right', ncol=2,
          facecolor=DARK3, edgecolor=BORDER, labelcolor=TEXT, fontsize=8)

ax.set_title(
    f'BSC TechSolve — Grafo Causal SEM (MLE)  |  '
    f'CFI={cfi:.3f}  RMSEA={rmsea:.3f}  SRMR={srmr:.3f}',
    color=TEXT, fontsize=12, pad=16
)
ax.set_xlim(-1.2, 9.2)
ax.set_ylim(-1.2, 6.5)
ax.axis('off')
plt.tight_layout()
plt.savefig('bsc_sem_graph.png', dpi=150,
            bbox_inches='tight', facecolor=DARK)
plt.show()